In [ ]:
from pathlib import Path
import os
import re
import time
import datetime
import shutil
import tiktoken
import boto3
import httpx
import pandas as pd
import pprint
import json
import pyarrow as pa
import pyarrow.parquet as pq
from tabulate import tabulate
from dotenv import load_dotenv
import aioboto3
import instructor
import asyncio
from pydantic import BaseModel, Field
from typing import Literal, Optional, List
import re
from lxml import etree
from xml.dom import minidom
from datetime import datetime
from bs4 import BeautifulSoup, Tag
from langchain_core.prompts import PromptTemplate
import os
import random
import string
import io
import gc
import traceback
import warnings
from contextlib import redirect_stdout
import copy
import fitz
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
os.makedirs("test_path/final_data", exist_ok=True)
os.makedirs("test_path/workings", exist_ok=True)

In [ ]:
def file_searcher(path):
    file_paths = []

    for file in path.rglob("*"):
        if file.is_file():
            file_paths.append(str(file))

    print("Total files found:", len(file_paths))
    return file_paths

def extract_text_from_pdf(pdf_paths):
    full_text = {}
    for i in pdf_paths:
        doc = fitz.open(i)
        full_text[i]=""
        for page in doc:
            full_text[i]+= page.get_text()

    return full_text

def chunk_by_file(file_dict):

    chunks = []

    for file_path, content in file_dict.items():
        chunks.append({
            "file_name": file_path,
            "content": content.strip()
        })

    return chunks

def create_vector_store(chunks):

    texts = [
        chunk["content"]
        for chunk in chunks
    ]

    embeddings = model.encode(texts)
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index, chunks

def search(query, index, metadata, top_k):

    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx, score in zip(indices[0], distances[0]):
        results.append({
            "file_name": metadata[idx]["file_name"],
            "content": metadata[idx]["content"],
            "score": float(score)
        })

    return results

In [ ]:
directory_path = Path("test_path")   # <-- Change this

path_list=file_searcher(directory_path)

In [ ]:
path_list

#### IDENTIFYING ITR FORM

In [ ]:
extracted_data = extract_text_from_pdf(path_list)

In [35]:
chunks = chunk_by_file(extracted_data)

In [ ]:
index , metadata = create_vector_store (chunks)

In [ ]:
result = search("Indian Income Tax Return, ITR-1", index, metadata, len(path_list))

In [ ]:
ITR_form = result[0]['file_name']

#### Creating Vector Store for ITR_Form Comparison

In [ ]:
path_list.remove(ITR_form)

In [ ]:
extracted_data_2 = extract_text_from_pdf(path_list)

In [ ]:
chunks = chunk_by_file(extracted_data_2)

In [ ]:
index , metadata = create_vector_store (chunks)

In [ ]:
result = search(ITR_form, index, metadata, len(extracted_data_2))

In [ ]:
result

#### File Segregation based on score

In [ ]:
for i in result:
    if i['score'] > 1.0:
        shutil.copy(i['file_name'], "test_path/workings")
    else:
        shutil.copy(i['file_name'], "test_path/final_data")

In [ ]:
print("Successfully Segregated the files")